In [ ]:
from src.utils import setup_client

url = setup_client()

### Loading dataset

In [ ]:
import pandas as pd

dataset_name = 'college_biology'
splits = {
    'test': f'{dataset_name}/test-00000-of-00001.parquet',
    'validation': f'{dataset_name}/validation-00000-of-00001.parquet',
    'dev': f'{dataset_name}/dev-00000-of-00001.parquet'
}
base_url = "hf://datasets/cais/mmlu/"
df_full = pd.read_parquet(base_url + splits["test"])


In [ ]:
df_full

In [ ]:
df_full.iloc[12]

In [ ]:
# Choose one question and query the model
question_choice = 12 
question = df_full.iloc[question_choice]['question']
choices = df_full.iloc[question_choice]['choices']
answer = df_full.iloc[question_choice]['answer']
# convert 0-based index to letter
answer = chr(answer + ord('A'))

# generate the prompt
choices_text = [f"{letter}. {text}" for letter, text in zip(['A', 'B', 'C', 'D'], choices)]
remark = "Please only answer with a single letter (A, B, C, or D) and nothing else."
prompt = f"{question}\n" + "\n".join(choices_text) + "\n" + remark
print(f"Designed prompt for question {question_choice}:\n")
print("-" * 50)
print(prompt)
print("-" * 50)
print(f"Correct answer: {answer}")


In [ ]:
import requests
def query_model(prompt, model_name, max_tokens, client):
    """
    Parameters
    ----------
    prompt     : str   — the input text
    model_name : str   — "gpt2", "llama-1b", or "llama-8b"
    max_tokens : int   — how many tokens to generate (capped at 200 by proxy)
    client     : str   — the proxy URL returned by setup_client()

    Returns
    -------
    dict with keys:
        "answer"      : str   — the generated text
        "logprobs"    : dict  — {token: logprob}
        "token_probs" : dict  — {token: probability}
    """
    system_prompt = """You are an answer key. You receive multiple choice questions and output only the answers.
                RULES:
                - Output one line per question.
                - Each line is exactly: the question number, a colon, a space, and a single capital letter (A, B, C, or D).
                - Output nothing else: no explanations, no restating the question, no extra words, no blank lines, no markdown.
                - Answer every question. If unsure, still pick the single most likely letter.

                FORMAT EXAMPLE (for 3 questions):
                1: A
                2: C
                3: B
                """
    
    response = requests.post(
        f"{client}/generate",
        json={
            "system_prompt": system_prompt,
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )

    if response.status_code == 429:
        print("⏳ Rate limit hit — wait a moment and try again.")
        return None

    if response.status_code == 400:
        print(f"❌ Bad request: {response.json().get('detail')}")
        return None


    response.raise_for_status()
    return response.json()


In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct" #"meta-llama/Llama-3.1-8B-Instruct" # "meta-llama/Llama-3.2-1B-Instruct" 

result = query_model(prompt, model_name, max_tokens=5, client=url)

print(f"Model's answer: {result['answer']}")
print("Token probabilities for each token in the answer:")
print(result['token_probs'])
print(f"Correct answer: {df_full.iloc[question_choice]['answer']}")


In [ ]:
models = [
    "meta-llama/Llama-3.1-8B-Instruct",   # current - already working
    "meta-llama/Llama-3.2-1B-Instruct",   # tiny
]

for model in models:
    result = query_model(prompt, model, max_tokens=5, client=url)
    print(f"Model: {model}")
    print(f"Model's answer: {result['answer']} | Correct answer: {answer}")
    print(f"Token probs for each token in the answer: {result['token_probs']}")
    print(f"--" * 20)

## Testing first 5 rows of dataset

In [ ]:
def batch_dataframe_into_prompts(df, batch_size, prefix=None, suffix=None):
    prompts = []
    correct_answers = []
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        corrent_answers_this_batch = {}
        if prefix:
            batch_prompt = prefix + "\n"
        else:
            batch_prompt = f"Answer the following {batch_size} multiple choice questions. \n"
            batch_prompt += "For each question, please only answer with a question number followed by a letter (A, B, C, or D) and nothing else.\n"
        
        for idx, row in batch.iterrows():
            question = row['question']
            choices = row['choices']
            choices_text = [f"{letter}. {text}" for letter, text in zip(['A', 'B', 'C', 'D'], choices)]
            prompt = f"{idx+1}: {question}\n" + "\n".join(choices_text) 
            batch_prompt += prompt + "\n\n"
            corrent_answers_this_batch[idx+1] = chr(row['answer'] + ord('A'))
        if suffix:
            batch_prompt += suffix
            
        prompts.append(batch_prompt)
        correct_answers.append(corrent_answers_this_batch)
    return prompts, correct_answers



In [ ]:
batch_size = 10
prompts, correct_answers = batch_dataframe_into_prompts(df_full, batch_size=batch_size)

In [ ]:
batch_to_query = 0
sample_prompt = prompts[batch_to_query]
sample_correct_answers = correct_answers[batch_to_query]

print(f"Sample prompt for batch {batch_to_query}:\n")
print("-" * 50)
print(sample_prompt)

In [ ]:
result = query_model(sample_prompt, model_name, max_tokens=5000, client=url)
print(f"Model's answer:\n{result['answer']}")

In [ ]:
def parse_answers(model_answer):
    answers_dict = {}
    lines = model_answer.strip().split("\n")
    for line in lines:
        if ": " in line:
            question_num, answer = line.split(": ", 1)
            answers_dict[question_num.strip()] = answer.strip()
    return answers_dict

def parse_answers_with_token_prob(result):
    answers_dict = {}
    lines = result['answer'].strip().split("\n")
    for line in lines:
        if ": " in line:
            question_num, answer = line.split(": ", 1)
            answers_dict[question_num.strip()] = {
                "answer": answer.strip(),
                "token_probs": result['token_probs']
            }
    return answers_dict
answers_dict = parse_answers(result['answer'])
print("Parsed answers:")
print(answers_dict)
# evaluate the answers

def evaluate_batched_answers(correct_answers_batch, answers_dict, verbose=False):
    correct_count = 0
    for question_num, correct_answer in correct_answers_batch.items():
        model_answer = answers_dict.get(str(question_num), None)
        if model_answer == correct_answer:
            correct_count += 1
        if verbose:
            print(f"Question {question_num}: Correct answer = {correct_answer}, Model answer = {model_answer}, {'✅' if model_answer == correct_answer else '❌'}")
    return correct_count / len(correct_answers_batch)
accuracy = evaluate_batched_answers(sample_correct_answers, answers_dict, verbose=True)

print(f"Model accuracy on {batch_size} questions: {accuracy:.2%}")

In [ ]:
# Compute model accuracy for the batch

print(f"Model accuracy on batch of {batch_size} questions: {accuracy:.2%}")

In [ ]:
import math
# limit dataset to first 5 rows, just to test
df = df_full.head(10).copy() 
batch_size = 10
# maps the integer answers from the data set, to the actual letters used in the prompt
ANSWER_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}


model_name = "meta-llama/Llama-3.2-1B-Instruct" 
max_output_tokens = 5 # only 5 as we just want one single letter as output from the model

results = [] # to store results of api calls

print(f"\nStarting API calls for {len(df)} questions in batches of {batch_size}...\n")
batched_prompts, batched_answers = batch_dataframe_into_prompts(df, batch_size=batch_size) 


# iterate through batched prompts and make API calls for each batch, then evaluate the results
for batch_index, (batch_prompt, correct_answers_batch) in enumerate(zip(batched_prompts, batched_answers)):

    # make API call and store log probabilities!
    try:
        result = query_model(batch_prompt, model_name, max_tokens=500, client=url)

        
        # Process the response of the model
        final_answer = result['answer']
        
        results.append({
            'id': batch_index,
            'question': batch_prompt,
            'correct_answer': correct_answers_batch,
            'model_answer': final_answer,
            'is_correct': final_answer == correct_answers_batch.get(str(batch_index+1), None),
            'chosen_logprob': chosen_logprob,
            'chosen_probability': safe_probability
        })

        answer_is_incorrect = final_answer != correct_answer_letter
        # prints where we are in the iteration and highlight incorrect answers with an asterisk
        if answer_is_incorrect:
            print(f"* Q {index+1}: Correct={correct_answer_letter}, Model={final_answer}, LogProb={chosen_logprob:.10f}, Prob={safe_probability*100:.10f}%")
        else:
            print(f"Q {index+1}: Correct={correct_answer_letter}, Model={final_answer}, LogProb={chosen_logprob:.10f}, Prob={safe_probability*100:.10f}%")

    except Exception as e:
        print(f"Error during API call for Q {index+1}: {e}. Skipping...")
        continue
        
print("\nAPI calls complete.")

# quick analysation of results
results_df = pd.DataFrame(results)
print("\n--- Results Summary (First 5 Questions) ---")
print(f"Total questions processed: {len(results_df)}")
print(f"Overall Accuracy: {results_df['is_correct'].mean():.6f}")


## Testing out the creation of first calibration plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Create calibration plot, using the Expected Calibration Error (ECE)
NUM_BINS = 10 
bin_edges = np.linspace(0.0, 1.0, NUM_BINS + 1) 

# ensure there is something in results_df
if len(results_df) == 0:
    print("error: results_df is empty.")
    exit()

# bin the confidence scores, using pd.cut (assigns data values to bins)
results_df['bin'] = pd.cut(
    results_df['chosen_probability'], 
    bins=bin_edges, 
    labels=False, 
    include_lowest=True # whether the first interval should be left-inclusive or not
)

# calculating metrics per bin
calibration_data = results_df.groupby('bin').agg(
    # calculate average predicted probability per bin
    avg_confidence=('chosen_probability', 'mean'),
    # calculate the actual accuracy, using the 'is_correct' boolean in the bin
    avg_accuracy=('is_correct', 'mean'),
    std_accuracy=('is_correct', 'std'), # standard deviation of accuracy in the bin, useful for error bars in the plot
    # count the samples in the bin
    count=('id', 'count') 
).reset_index() # reset_index moves the current index (the values of the 'bin' column) out of the index position and converts it back into a standard df column named 'bin'

# compute standard error of accuracy for error bars in the plot
calibration_data['accuracy_se'] = calibration_data['std_accuracy'] / np.sqrt(calibration_data['count'])
# replace nan values with 0s
calibration_data = calibration_data.fillna(0)

# calculate the center of each bin
def calculate_bin_center(bin_index):
    bin_index = int(bin_index) 
    return (bin_edges[bin_index] + bin_edges[bin_index + 1]) / 2.0

# get correct center for each row
calibration_data['bin_center'] = calibration_data['bin'].apply(calculate_bin_center)


# Calculate ECE!
# first the weights of each bin: (number of samples in bin) / (total number of samples)
calibration_data['weight'] = calibration_data['count'] / calibration_data['count'].sum()

# ECE
calibration_data['ece_term'] = calibration_data['weight'] * np.abs(
    calibration_data['avg_accuracy'] - calibration_data['avg_confidence']
)
ece = calibration_data['ece_term'].sum()

print(f"\nExpected Calibration Error (ECE): {ece:.6f}")
print("-" * 50)


# PLOT
fig, ax = plt.subplots(figsize=(8, 8))

# plot calibration curve (accuracy vs confidence)
ax.bar(
    calibration_data['bin_center'], 
    calibration_data['avg_accuracy'], 
    width=(1/NUM_BINS) * 0.9, # Bar width slightly less than bin width
    color='skyblue', 
    edgecolor='black',
    label='Actual Accuracy'
)

# plot difference between acc and conf
ax.scatter(
    calibration_data['bin_center'],
    calibration_data['avg_confidence'],
    marker='o',
    color='red',
    s=100,
    label='Average Predicted Confidence'
)

ax.errorbar(
    calibration_data['bin_center'],
    calibration_data['avg_accuracy'],
    yerr=calibration_data['accuracy_se'],
    fmt='none',
    ecolor='gray',
    alpha=0.7,
    capsize=5
)

# plot desired perfect calibration line
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')

# Set labels and title
ax.set_xlabel('Average Predicted Confidence (Bin Center)')
ax.set_ylabel('Actual Accuracy in Bin')
ax.set_title(f'Calibration Plot for Gemini-2.0-Flash (Subject: {dataset_name})')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, linestyle=':', alpha=0.6)

# Add ECE text to the plot
ax.text(0.1, 0.9, f'ECE: {ece:.4f}', transform=ax.transAxes, fontsize=12, 
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

# Display the plot
plt.show()

In [ ]:
# Plot reliability curve with just scatter points and a dashed line for perfect calibration
fig, ax = plt.subplots(figsize=(8, 8))
# plot the reliability curve (accuracy vs confidence)
ax.scatter(
    calibration_data['bin_center'],
    calibration_data['avg_accuracy'],
    marker='o',
    color='blue',
    s=100,
    label='Actual Accuracy'
)
# plot the perfect calibration line
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
# Set labels and title
ax.set_xlabel('Average Predicted Confidence (Bin Center)')
ax.set_ylabel('Actual Accuracy in Bin')


In [ ]:
results_df

## Defining calibration plot function


In [ ]:


# Constants outside the function
ANSWER_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # "meta-llama/Llama-3.1-8B-Instruct" # "meta-llama/Llama-3.2-1B-Instruct"
MAX_OUTPUT_TOKENS = 5 
BASE_URL = "hf://datasets/cais/mmlu/"


def calibration_plot(dataset_name: str, number_of_bins: int):
    """
    Generates a calibration plot for the specified MMLU dataset using the Gemini API.

    Args:
        dataset_name (str): The name of the MMLU dataset subset (e.g., 'college_biology').
        number_of_bins (int): The number of bins to use for the calibration plot.
    """
    import os
    from src.utils import setup_client
    from tqdm import tqdm
    from time import sleep
    # DATA LOADING
    
    splits = {
        'test': f'{dataset_name}/test-00000-of-00001.parquet',
        'validation': f'{dataset_name}/validation-00000-of-00001.parquet',
        'dev': f'{dataset_name}/dev-00000-of-00001.parquet'
    }
    
    try:
        # Load the FULL dataset for proper calibration (removed .head(5))
        df = pd.read_parquet(BASE_URL + splits["test"])
    except Exception as e:
        print(f"Error loading dataset '{dataset_name}': {e}")
        return

    
    # API SET UP
    
    try:
        # Client needs to be accessible, assuming it's initialized globally or passed in.
        # Since your original code initializes it, we'll assume 'client' is defined here.
        client = setup_client()
        print("HF API Client initialized.")
    except Exception as e:
        print(f"Error when initializing client: {e}")
        return # Use return instead of exit() inside a function

    
    # API CALLS
    
    results = [] # to store results of api calls
    num_questions = len(df)
    print(f"\nStarting API calls for {num_questions} questions in '{dataset_name}'...")

    # iterate through dataframe
    for index, row in tqdm(df.iterrows(), total=num_questions, desc="Answering..."):
        # map the integer answer to the actual letter using the  ANSWER_MAP dict
        sleep(0.5)
        correct_answer_index = row['answer'] 
        correct_answer_letter = ANSWER_MAP.get(correct_answer_index, 'UNKNOWN')
        
        # skip if answer is unknown
        if correct_answer_letter == 'UNKNOWN':
            continue

        question_text = row['question'] # the actual question in the dataset
        choices = row['choices'] # the multiple choice options for the answer
        
        # make a nice format of the options so that the prompt is correct and easy to read
        option_lines = [f"{letter}. {text}" for letter, text in zip(['A', 'B', 'C', 'D'], choices)]
        
        # construct the final structured prompt
        full_prompt = (
            f"{question_text}\n" + 
            "\n".join(option_lines) +
            "\n\nPlease only answer with a single letter (A, B, C, or D) and nothing else."
        )
        
        # make API call and store log probabilities!
        try:
            result = query_model(full_prompt, MODEL_NAME, max_tokens=MAX_OUTPUT_TOKENS, client=client)
            # Process the response of the model
            final_answer = result['answer']
            logprob_content = result['logprobs']
            chosen_logprob = result['token_probs'][final_answer] if final_answer in result['token_probs'] else None
            safe_probability = min(chosen_logprob, 1.0) if chosen_logprob is not None else None
        
            results.append({
                'id': index,
                'correct_answer': correct_answer_letter,
                'model_answer': final_answer,
                'is_correct': final_answer == correct_answer_letter,
                'chosen_logprob': chosen_logprob,
                'chosen_probability': safe_probability
            })
            
            #if final_answer is None or chosen_logprob is None or safe_probability is None:
            #    print(f"Q {index+1}: WARNING: Model generated no tokens.")

        except Exception as e:
            # catch errors to prevent one single error of stopping entire function
            print(f"Error during API call for Q {index+1}: {e}. Skipping...")
            continue
            
    print("\nAPI calls complete.")

    
    # CALIBRATION
    
    results_df = pd.DataFrame(results)
    
    if len(results_df) == 0:
        print("error: results_df is empty after API calls.")
        return

    # quick analysation of results
    print("\n--- Results Summary ---")
    print(f"Total questions processed: {len(results_df)}")
    print(f"Overall Accuracy: {results_df['is_correct'].mean():.6f}")
    
    # Create calibration plot, using the Expected Calibration Error (ECE)
    bin_edges = np.linspace(0.0, 1.0, number_of_bins + 1) 

    # bin the confidence scores, using pd.cut
    results_df['bin'] = pd.cut(
        results_df['chosen_probability'], 
        bins=bin_edges, 
        labels=False, 
        include_lowest=True
    )

    # calculating metrics per bin
    calibration_data = results_df.groupby('bin').agg(
        avg_confidence=('chosen_probability', 'mean'),
        avg_accuracy=('is_correct', 'mean'),
        std_accuracy=('is_correct', 'std'), # standard deviation of accuracy in the bin, useful for error bars in the plot
        count=('id', 'count') 
    ).reset_index()

    # compute standard error of accuracy for error bars in the plot
    calibration_data['accuracy_se'] = calibration_data['std_accuracy'] / np.sqrt(calibration_data['count'])

    # replace nan values with 0s
    calibration_data = calibration_data.fillna(0)

    # calculate the center of each bin
    def calculate_bin_center(bin_index):
        bin_index = int(bin_index) 
        return (bin_edges[bin_index] + bin_edges[bin_index + 1]) / 2.0

    # get correct center for each row
    calibration_data['bin_center'] = calibration_data['bin'].apply(calculate_bin_center)


    # Calculate ECE!
    # first the weights of each bin
    calibration_data['weight'] = calibration_data['count'] / calibration_data['count'].sum()

    # ECE
    calibration_data['ece_term'] = calibration_data['weight'] * np.abs(
        calibration_data['avg_accuracy'] - calibration_data['avg_confidence']
    )
    ece = calibration_data['ece_term'].sum()

    print(f"\nExpected Calibration Error (ECE): {ece:.6f}")
    print("-" * 50)


    # PLOTTING
    
    fig, ax = plt.subplots(figsize=(8, 8))

    # plotting actual accuracy 
    ax.scatter(
        calibration_data['bin_center'],
        calibration_data['avg_accuracy'],
        marker='o',
        color='blue',
        s=100,
        label='Actual Accuracy'
    )

    ax.errorbar(
        calibration_data['bin_center'],
        calibration_data['avg_accuracy'],
        yerr=calibration_data['accuracy_se'],
        fmt='none',
        ecolor='gray',
        alpha=0.7,
        capsize=5
    )


    # Print number of samples in each bin
    for _, row in calibration_data.iterrows():
        count = row['count']
        center = row['bin_center']
        
        if count >= 0:
            # Set y-coordinate to a small fixed value (e.g., 0.02) just above the axis.
            # Use 'va="bottom"' to ensure the text starts at 0.02 and doesn't overlap the axis.
            ax.text(
                center, 
                0.02, # Fixed y-coordinate close to the bottom (X-axis)
                f'N={int(count)}',
                ha='center',       # Horizontal alignment (centered on the bar)
                va='bottom',       # Vertical alignment (text base starts at y=0.02)
                fontsize=9,        
                color='black',
                weight='bold'      # Added bolding for better visibility against the blue bar
            )
            

    # plot desired perfect calibration line
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')

    # Set labels and title
    ax.set_xlabel('Average Predicted Confidence (Bin Center)')
    ax.set_ylabel('Actual Accuracy in Bin')
    ax.set_title(f'Calibration Plot for {MODEL_NAME} (Subject: {dataset_name}. Ntot={len(df)}.)')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.6)

    # Add ECE text to the plot
    ax.text(0.1, 0.9, f'ECE: {ece:.4f}', transform=ax.transAxes, fontsize=12, 
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

    # display the plot
    plt.show()




In [ ]:
number_of_bins = 10
data_sets = ['college_biology', 'college_medicine', 'conceptual_physics', 'econometrics', 'high_school_biology', 'high_school_psychology',
             'professional_medicine', 'professional_psychology']
#data_sets = ["professional_medicine", "professional_psychology"]
for data_set in data_sets:
    calibration_plot(data_set, number_of_bins)
